<a href="https://colab.research.google.com/github/Minh-Quan-hub/build-ai-agents-and-chatbots-118s/blob/Agentic-AI-in-Supply-Chain/Agentic_AI_in_Supply_Chain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **1. Inventory replenishment agent**

In [1]:
import math
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm

sales = pd.read_csv("sales.csv", dtype={"sku": str}, parse_dates=["date"])
inventory = pd.read_csv("inventory.csv", dtype={"sku": str})
params = pd.read_csv("params.csv", dtype={"sku": str})

# Aggregate duplicate transaction-day rows, then restore zero-demand dates.
daily = sales.groupby(["date", "sku"], as_index=False)["qty_sold"].sum()
dates = pd.date_range(daily.date.min(), daily.date.max(), freq="D")
skus = params.sku.tolist()
full_index = pd.MultiIndex.from_product([dates, skus], names=["date", "sku"])
demand = daily.set_index(["date", "sku"])["qty_sold"].reindex(full_index, fill_value=0).astype(int)
assert set(skus) == set(inventory.sku) == set(sales.sku)
assert inventory.sku.is_unique and params.sku.is_unique
assert (demand >= 0).all() and (inventory.opening_stock >= 0).all()
assert params.lead_time_days.ge(1).all() and params.min_order_qty.ge(1).all()
assert params.service_level.between(0.5, 0.999).all()
assert params[["unit_cost", "holding_cost_per_day", "stockout_cost"]].ge(0).all().all()
assert params[["unit_cost", "holding_cost_per_day", "stockout_cost"]].ge(0).all().all()
print(f"Loaded {len(dates)} days, {len(skus)} SKUs, {len(demand):,} daily rows.")
display(demand.rename("qty_sold").reset_index().head(9))

Loaded 90 days, 3 SKUs, 270 daily rows.


,date,sku,qty_sold
0,2011-01-01,22720,0
1,2011-01-01,22666,0
2,2011-01-01,22960,0
3,2011-01-02,22720,0
4,2011-01-02,22666,0
5,2011-01-02,22960,0
6,2011-01-03,22720,0
7,2011-01-03,22666,0
8,2011-01-03,22960,0


### **2. Forecast, safety stock, and agent policy**

In [2]:
import math
from collections import defaultdict, deque
from dataclasses import dataclass, field

from scipy.stats import norm

ALPHA = 0.30
ERROR_WINDOW = 28
HISTORY_WINDOW = 56
FIXED_PO_COST = 0.0
REVIEW_PERIOD_DAYS = 1

@dataclass
class ForecastState:
    level: float = 0.0
    initialized: bool = False
    errors: deque = field(default_factory=lambda: deque(maxlen=HISTORY_WINDOW))

    def update(self, actual, prediction):
        self.errors.append(float(actual - prediction))
        self.level = (ALPHA * actual + (1 - ALPHA) * self.level) if self.initialized else float(actual)
        self.initialized = True

    def safety_stock(self, protection, service_level, enhanced):
        recent = np.array(list(self.errors)[-ERROR_WINDOW:], dtype=float)
        rmse = float(np.sqrt(np.mean(recent**2))) if recent.size else 0.0
        normal_buffer = float(norm.ppf(service_level) * rmse * math.sqrt(protection))
        if not enhanced or len(self.errors) < 3 * protection:
            return normal_buffer, rmse
        errors = np.array(self.errors, dtype=float)
        window_sums = np.convolve(errors, np.ones(protection), mode="valid")
        empirical_buffer = float(np.quantile(window_sums, service_level))
        return max(normal_buffer, empirical_buffer, 0.0), rmse

class ReplenishmentAgent:
    def decide(self, date, on_hand, pending_qty, forecast, p, enhanced):
        protection = int(p.lead_time_days) + REVIEW_PERIOD_DAYS
        projected_demand = forecast.level * protection
        shortage = float(p.stockout_cost)
        carry = float(p.holding_cost_per_day) * protection
        cost_based_service = shortage / (shortage + carry) if shortage + carry else 0.0
        risk_quantile = min(0.999, max(float(p.service_level), cost_based_service)) if enhanced else float(p.service_level)
        safety, rmse = forecast.safety_stock(protection, risk_quantile, enhanced)
        target = projected_demand + safety
        position = on_hand + pending_qty
        gap = target - position
        qty = max(int(p.min_order_qty), math.ceil(gap)) if gap > 1e-9 else 0
        return qty, position, target, safety, rmse, projected_demand, risk_quantile

def simulate(demand, inventory, params, enhanced=True):
    policy = params.set_index("sku")
    stock = inventory.set_index("sku").opening_stock.astype(int).to_dict()
    forecasts = {sku: ForecastState() for sku in policy.index}
    pending = defaultdict(int)  # (arrival_date, SKU) -> units, including multiple POs
    agent = ReplenishmentAgent()
    rows = []

    for date in demand.index.get_level_values("date").unique():
        for sku in policy.index:
            p = policy.loc[sku]
            received = pending.pop((date, sku), 0)
            stock[sku] += received
            opening = stock[sku]
            pipeline = sum(q for (due, item), q in pending.items() if item == sku)
            state = forecasts[sku]
            today_prediction = state.level
            qty, position, target, safety, rmse, projected, risk_quantile = agent.decide(
                date, opening, pipeline, state, p, enhanced
            )
            if qty:
                eta = date + pd.Timedelta(days=int(p.lead_time_days))
                pending[(eta, sku)] += qty
            requested = int(demand.loc[(date, sku)])
            filled = min(opening, requested)
            lost = requested - filled
            stock[sku] -= filled
            rows.append({
                "date": date, "sku": sku, "received": received,
                "opening_on_hand": opening,
                "forecast_today": round(today_prediction, 2),
                "forecast_protection": round(projected, 2),
                "forecast_rmse": round(rmse, 2),
                "risk_quantile": round(risk_quantile, 4),
                "safety_stock": round(safety, 2),
                "inventory_position": position, "target_position": round(target, 2),
                "action": "ORDER" if qty else "WAIT", "order_qty": qty,
                "eta": eta.date().isoformat() if qty else "",
                "open_po_qty": pipeline + qty,
                "demand": requested, "fulfilled": filled, "lost": lost,
                "closing_on_hand": stock[sku],
                "holding_cost": stock[sku] * float(p.holding_cost_per_day),
                "stockout_cost": lost * float(p.stockout_cost),
                "order_cost": FIXED_PO_COST if qty else 0.0,
                "purchase_spend": qty * float(p.unit_cost),
                "rationale": (
                    f"Position {position} < target {target:.1f} "
                    f"(forecast {projected:.1f} + safety {safety:.1f} at {risk_quantile:.1%}); "
                    f"order {qty}, minimum {int(p.min_order_qty)}"
                    if qty else
                    f"Position {position} >= target {target:.1f} "
                    f"(forecast {projected:.1f} + safety {safety:.1f} at {risk_quantile:.1%}); wait"
                ),
            })
            state.update(requested, today_prediction)  # no future information
    return pd.DataFrame(rows)

baseline_log = simulate(demand, inventory, params, enhanced=False)
log = simulate(demand, inventory, params, enhanced=True)

In [7]:
from html import escape
from IPython.display import HTML, display

def summarize(records):
    costs = ["holding_cost", "stockout_cost", "order_cost", "purchase_spend"]
    result = records.groupby("sku", as_index=False).agg(
        requested=("demand", "sum"), fulfilled=("fulfilled", "sum"),
        lost=("lost", "sum"), stockout_days=("lost", lambda s: int((s > 0).sum())),
        purchase_orders=("order_qty", lambda s: int((s > 0).sum())),
        closing_stock=("closing_on_hand", "last"),
        open_po_qty=("open_po_qty", "last"),
        **{c: (c, "sum") for c in costs},
    )
    total = {"sku": "ALL"}
    for c in result.columns.drop("sku"):
        total[c] = result[c].sum()
    result = pd.concat([result, pd.DataFrame([total])], ignore_index=True)
    result["fill_rate"] = result.fulfilled / result.requested
    result["operating_cost"] = result.holding_cost + result.stockout_cost + result.order_cost
    return result

baseline_summary = summarize(baseline_log)
summary = summarize(log)
overall = summary.iloc[-1]
old = baseline_summary.iloc[-1]
money = lambda value: f"${value:,.2f}"

css = """
<style>
.ir-report {font: 14px/1.45 -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
            color: #263b43; max-width: 1100px;}
.ir-report h2 {font-size: 21px; margin: 22px 0 7px; color: #163744;}
.ir-report .ir-note {color: #607780; margin: 5px 0 14px;}
.ir-report .ir-cards {display: flex; flex-wrap: wrap; gap: 12px; margin: 14px 0 18px;}
.ir-report .ir-card {background: #f1f7f5; border-left: 4px solid #34856e;
                     padding: 10px 15px; min-width: 155px; flex: 1; border-radius: 6px;}
.ir-report .ir-card strong {display: block; font-size: 22px; color: #1d5c4c;}
.ir-report .ir-card small {color: #546f75;}
.ir-report .ir-wrap {overflow-x: auto; border: 1px solid #dce7e5;
                     border-radius: 7px; margin: 9px 0 17px;}
.ir-report table {border-collapse: collapse; min-width: 830px; width: 100%; font-size: 13px;}
.ir-report th {background: #e8f0ef; text-align: left; color: #284d50;
               font-weight: 650; position: sticky; top: 0; z-index: 1;}
.ir-report th, .ir-report td {padding: 8px 10px; border-bottom: 1px solid #e6eeed; white-space: nowrap;}
.ir-report tr:last-child td {border-bottom: 0;}
.ir-report tbody tr:nth-child(even) {background: #f8fbfa;}
.ir-report tbody tr.ir-total {background: #edf5f1; font-weight: 700;}
.ir-report tbody tr.ir-loss {background: #fff4f0;}
.ir-report .ir-money, .ir-report .ir-num {text-align: right; font-variant-numeric: tabular-nums;}
.ir-report .ir-badge {display: inline-block; padding: 2px 7px; border-radius: 4px;
                      font-weight: 650; font-size: 12px;}
.ir-report .ir-order {color: #135b46; background: #dcefe6;}
.ir-report .ir-wait {color: #526973; background: #edf2f2;}
.ir-report .ir-lost {color: #a13525; font-weight: 700;}
.ir-report details {border: 1px solid #dce7e5; border-radius: 7px;
                    margin: 9px 0; background: #fff;}
.ir-report summary {padding: 11px 14px; cursor: pointer; font-weight: 650;
                    background: #f2f7f6; border-radius: 7px;}
.ir-report details[open] summary {border-bottom: 1px solid #dce7e5;}
.ir-report .ir-scroll {max-height: 440px; overflow: auto;}
.ir-report .ir-scroll table {min-width: 1120px;}
.ir-report .ir-why {white-space: normal; min-width: 185px;}
</style>
"""

cards = f"""
<div class="ir-cards">
  <div class="ir-card"><small>Fill rate</small><strong>{overall.fill_rate:.1%}</strong><small>{int(overall.fulfilled):,} of {int(overall.requested):,} units filled</small></div>
  <div class="ir-card"><small>Operating cost</small><strong>{money(overall.operating_cost)}</strong><small>Holding + shortage + PO fees</small></div>
  <div class="ir-card"><small>Lost demand</small><strong>{int(overall.lost):,}</strong><small>Across {int(overall.stockout_days)} SKU-days</small></div>
  <div class="ir-card"><small>Purchase orders</small><strong>{int(overall.purchase_orders)}</strong><small>{int(overall.open_po_qty)} units still in transit at end</small></div>
</div>
"""

def table(headers, body, row_classes=None):
    row_classes = row_classes or [""] * len(body)
    head = "".join(f"<th>{escape(label)}</th>" for label in headers)
    rows = "".join(
        f'<tr class="{cls}">' + "".join(f"<td>{value}</td>" for value in values) + "</tr>"
        for cls, values in zip(row_classes, body)
    )
    return f"<table><thead><tr>{head}</tr></thead><tbody>{rows}</tbody></table>"

def number(value):
    return f'<span class="ir-num">{int(value):,}</span>'

def currency(value):
    return f'<span class="ir-money">{money(value)}</span>'

sku_rows = [
    [escape(str(row.sku)), f"{row.fill_rate:.1%}", number(row.lost),
     number(row.stockout_days), number(row.purchase_orders),
     currency(row.holding_cost), currency(row.stockout_cost),
     currency(row.operating_cost), number(row.closing_stock), number(row.open_po_qty)]
    for row in summary.itertuples(index=False)
]
sku_table = table(
    ["SKU", "Fill rate", "Lost units", "Stockout days", "POs", "Holding",
     "Stockout", "Total cost", "Closing stock", "In transit"],
    sku_rows, ["", "", "", "ir-total"]
)

comparison_rows = [
    [escape(name), f"{row.fill_rate:.1%}", number(row.lost),
     number(row.stockout_days), number(row.purchase_orders),
     currency(row.holding_cost), currency(row.stockout_cost), currency(row.operating_cost)]
    for name, row in [("Original EWMA", old), ("Enhanced agent", overall)]
]
comparison_table = table(
    ["Policy", "Fill rate", "Lost units", "Stockout days", "POs", "Holding",
     "Stockout", "Total cost"], comparison_rows, ["", "ir-total"]
)

report = [css, '<div class="ir-report">',
          '<h2>Inventory simulation · 90 days </h2>', cards,
          '<h2>Results by SKU</h2><div class="ir-wrap">', sku_table, '</div>',
          '<h2>Policy comparison</h2><div class="ir-wrap">', comparison_table, '</div>',
          f'<p class="ir-note">On this sample, the enhanced agent saves '
          f'{money(old.operating_cost - overall.operating_cost)} in modeled operating cost. '
          'Costs use illustrative inputs; procurement spend is reported.</p>',
          '<h2>Daily decisions</h2>',
          '<p class="ir-note">Expand a month and scroll to see every SKU-day. ']

for month, month_log in log.groupby(log.date.dt.to_period("M"), sort=True):
    daily_rows, row_classes = [], []
    for row in month_log.itertuples(index=False):
        short = (f"Below target by {row.target_position - row.inventory_position:.0f}; "
                 f"min {int(params.set_index('sku').loc[row.sku, 'min_order_qty'])}, "
                 f"due {escape(row.eta)}" if row.order_qty else
                 f"At/above target by {row.inventory_position - row.target_position:.0f}; hold")
        badge = (f'<span class="ir-badge ir-order">Order {row.order_qty}</span>' if row.order_qty
                 else '<span class="ir-badge ir-wait">Wait</span>')
        lost = f'<span class="ir-lost">{row.lost}</span>' if row.lost else '0'
        daily_rows.append([
            row.date.strftime("%b %d"), escape(row.sku), badge,
            f"{row.opening_on_hand:,}" + (f" (+{row.received:,})" if row.received else ""),
            f"{row.forecast_protection:.0f} + {row.safety_stock:.0f}",
            f"{row.inventory_position:,} / {row.target_position:.0f}",
            f"{row.fulfilled:,} / {row.demand:,}", lost,
            f"{row.closing_on_hand:,}", f'<span class="ir-why">{short}</span>'
        ])
        row_classes.append("ir-loss" if row.lost else "")
    daily_table = table(
        ["Day", "SKU", "Decision", "Stock now", "Forecast + safety", "Position / target",
         "Filled / demand", "Lost", "End stock", "Reason"], daily_rows, row_classes
    )
    orders = int(month_log.order_qty.gt(0).sum())
    loss = int(month_log.lost.sum())
    report.append(f'<details><summary>{month.strftime("%B %Y")} · '
                  f'{len(month_log)} SKU-days · {orders} orders · {loss} lost units'
                  f'</summary><div class="ir-scroll">{daily_table}</div></details>')

report.append('</div>')
display(HTML("".join(report)))

# Full, unrounded decision data is available for sorting/filtering outside the notebook.
log.to_csv("agent_log.csv", index=False)
summary.to_csv("agent_summary.csv", index=False)

PRINT_TEXT_LOG = False
if PRINT_TEXT_LOG:
    for row in log.itertuples(index=False):
        decision = f"ORDER {row.order_qty} (due {row.eta})" if row.order_qty else "WAIT"
        print(f"{row.date:%Y-%m-%d} | {row.sku} | {decision} | "
              f"position {row.inventory_position} / target {row.target_position:.0f} | "
              f"demand {row.demand}, lost {row.lost}, end stock {row.closing_on_hand}")